# 12 infercnvpy QC — canonical historical, memory-safe

This notebook reproduces the archived executed CNV visual-QC notebook
`12_CNV_visual_QC_and_interpretation_HVG8000_r0_6.ipynb` on the canonical
notebook-11 outputs.

The scientific visual-QC logic is preserved. The execution is made safe for the
current Windows / 16-GB-RAM system by first building lightweight NVMe cache
H5ADs containing only `obs`, `var`, `uns`, `X_cnv`, and `X_umap`. The full
15,163-gene expression matrix `X` is never loaded by the QC cells.

Historical structural checkpoints are enforced at the end of each stage.


# 28 — CNV visual QC and interpretation after infercnvpy

**Goal:** visually and quantitatively inspect infercnvpy CNV results across all
five CNV reference strategies. This notebook does **not** make a final malignant
call.

Archived executed checkpoints:

```text
combined_run_summary            (5, 18)
combined_cluster_summary        (30, 30)
consensus_calls                 (7, 11)
infercnvpy output manifest      (31, 4)

strategy visual summary         (5, 22)
cluster score summary           (30, 8)
chromosome signal summary       (0, 0)
interpretation table            (7, 17)
figure manifest                 (65, 5)
visual-QC output manifest       (6, 4)
```

Historical figures: 20 transcriptomic UMAPs, 30 infercnvpy chromosome
heatmap/summary plots, and 15 custom mean-CNV heatmaps.


In [1]:
# =========================
# Imports and configuration
# =========================

from pathlib import Path

import os
import gc
import copy
import warnings

warnings.filterwarnings("ignore")

import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt
import scipy.sparse as sp

try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem

try:
    import infercnvpy as cnv
    HAS_INFERCNVPY = True
except Exception as e:
    HAS_INFERCNVPY = False
    print("[WARNING] infercnvpy could not be imported:", repr(e))


def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        _find_repository_root(),
    )
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "interim"
FIGURES_DIR = PROJECT_DIR / "figures"

CNV_INPUT_DIR = RAW_DIR / "integrated" / "CNV_inputs"

# Exact canonical notebook-11 publication directory.
CNV_RESULT_DIR = (
    RAW_DIR
    / "integrated"
    / "CNV_results"
    / "infercnvpy_HVG8000_r0_6"
)

CNV_FIG_DIR = FIGURES_DIR / "CNV_visual_QC_HVG8000_r0_6"

for d in [
    METADATA_DIR,
    CNV_RESULT_DIR,
    CNV_FIG_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )

RUN_SUFFIX = "HVG8000_r0_6"

RUN_MANIFEST_TSV = (
    METADATA_DIR
    / f"infercnvpy_strategy_run_manifest_{RUN_SUFFIX}.tsv"
)
COMBINED_RUN_SUMMARY_TSV = METADATA_DIR / f"infercnvpy_combined_run_summary_{RUN_SUFFIX}.tsv"
COMBINED_CLUSTER_SUMMARY_TSV = METADATA_DIR / f"infercnvpy_combined_cluster_summary_{RUN_SUFFIX}.tsv"
CONSENSUS_CALLS_TSV = METADATA_DIR / f"infercnvpy_consensus_cluster_calls_{RUN_SUFFIX}.tsv"
INFERCNVPY_OUTPUT_MANIFEST_TSV = METADATA_DIR / f"infercnvpy_output_manifest_{RUN_SUFFIX}.tsv"

FIGURE_MANIFEST_TSV = METADATA_DIR / f"CNV_visual_QC_figure_manifest_{RUN_SUFFIX}.tsv"
STRATEGY_VISUAL_SUMMARY_TSV = METADATA_DIR / f"CNV_visual_QC_strategy_summary_{RUN_SUFFIX}.tsv"
CLUSTER_VISUAL_SUMMARY_TSV = METADATA_DIR / f"CNV_visual_QC_cluster_score_summary_{RUN_SUFFIX}.tsv"
CHROMOSOME_SIGNAL_TSV = METADATA_DIR / f"CNV_visual_QC_chromosome_signal_summary_{RUN_SUFFIX}.tsv"
INTERPRETATION_TSV = METADATA_DIR / f"CNV_visual_QC_interpretation_table_{RUN_SUFFIX}.tsv"
OUTPUT_MANIFEST_TSV = METADATA_DIR / f"CNV_visual_QC_output_manifest_{RUN_SUFFIX}.tsv"


def _default_scratch_root():
    configured = os.environ.get(
        "GLIOMA_MAIN12_SCRATCH_ROOT"
    )

    if configured:
        return (
            Path(configured)
            .expanduser()
            .resolve()
        )

    if os.name == "nt" and Path("E:/").exists():
        return Path(
            r"E:\glioma-cnv-scratch\main_analysis_12"
        )

    return (
        PROJECT_DIR
        / ".scratch"
        / "main_analysis_12"
    ).resolve()


SCRATCH_ROOT = _default_scratch_root()
SCRATCH_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


HISTORICAL_QC_STRATEGY_ORDER = [
    "core_conservative",
    "core_plus_lymphoid",
    "extended_myeloid",
    "review_ambiguous",
    "review_cycling",
]

EXPECTED_STRATEGY_CELLS = {
    "core_conservative": 32640,
    "core_plus_lymphoid": 38807,
    "extended_myeloid": 45000,
    "review_ambiguous": 19443,
    "review_cycling": 14640,
}

EXPECTED_STRATEGY_CLUSTER_ROWS = {
    "core_conservative": 6,
    "core_plus_lymphoid": 8,
    "extended_myeloid": 9,
    "review_ambiguous": 4,
    "review_cycling": 3,
}

EXPECTED_N_GENES = 15163
EXPECTED_N_CNV_WINDOWS = 1256

EXPECTED_COMBINED_RUN_SHAPE = (5, 18)
EXPECTED_COMBINED_CLUSTER_SHAPE = (30, 30)
EXPECTED_CONSENSUS_SHAPE = (7, 11)
EXPECTED_INFER_MANIFEST_SHAPE = (31, 4)

EXPECTED_DISCOVERY_SHAPE = (5, 4)
EXPECTED_STRATEGY_VISUAL_SHAPE = (5, 22)
EXPECTED_CLUSTER_SCORE_SHAPE = (30, 8)
EXPECTED_CHROMOSOME_SIGNAL_SHAPE = (0, 0)
EXPECTED_INTERPRETATION_SHAPE = (7, 17)
EXPECTED_FIGURE_MANIFEST_SHAPE = (65, 5)
EXPECTED_OUTPUT_MANIFEST_SHAPE = (6, 4)

EXPECTED_QUERY_REVIEW_CLUSTERS = {
    "0", "2", "4", "7", "9", "11", "12"
}


print("PROJECT_DIR:", PROJECT_DIR)
print("CNV_RESULT_DIR:", CNV_RESULT_DIR)
print("CNV_FIG_DIR:", CNV_FIG_DIR)
print("SCRATCH_ROOT:", SCRATCH_ROOT)
print("HAS_INFERCNVPY:", HAS_INFERCNVPY)

PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
CNV_RESULT_DIR: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\infercnvpy_HVG8000_r0_6
CNV_FIG_DIR: G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6
SCRATCH_ROOT: E:\glioma-cnv-scratch\main_analysis_12
HAS_INFERCNVPY: True


In [2]:
# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def safe_read_tsv(path, required=True):
    path = Path(path)

    if path.exists():
        return pd.read_csv(path, sep="\t")

    if required:
        raise FileNotFoundError(path)

    print(f"[MISSING] {path}")
    return pd.DataFrame()


def sanitize_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace(":", "_")
    )


def finite_summary(x):
    x = np.asarray(x).ravel()
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return {
            "min": np.nan,
            "q05": np.nan,
            "q25": np.nan,
            "median": np.nan,
            "mean": np.nan,
            "q75": np.nan,
            "q95": np.nan,
            "max": np.nan,
            "sd": np.nan,
        }

    return {
        "min": float(np.min(x)),
        "q05": float(np.quantile(x, 0.05)),
        "q25": float(np.quantile(x, 0.25)),
        "median": float(np.median(x)),
        "mean": float(np.mean(x)),
        "q75": float(np.quantile(x, 0.75)),
        "q95": float(np.quantile(x, 0.95)),
        "max": float(np.max(x)),
        "sd": float(np.std(x)),
    }


def detect_cnv_rep(adata):
    preferred = [
        "X_cnv",
        "cnv",
        "X_infercnv",
        "X_CNV",
    ]

    for k in preferred:
        if k in adata.obsm:
            return k

    cnv_like = [
        k
        for k
        in adata.obsm.keys()
        if "cnv" in str(k).lower()
    ]

    if cnv_like:
        return cnv_like[0]

    return None


def detect_cnv_score_col(adata):
    preferred = [
        "cnv_score",
        "CNV_score",
        "infercnvpy_cnv_score",
    ]

    for c in preferred:
        if c in adata.obs.columns:
            return c

    score_like = [
        c
        for c
        in adata.obs.columns
        if (
            "cnv" in str(c).lower()
            and "score" in str(c).lower()
        )
    ]

    if score_like:
        return score_like[0]

    return None


def ensure_cnv_score(
    adata,
    cnv_rep_key,
    score_key="cnv_score_visual",
):
    if score_key in adata.obs.columns:
        return score_key

    if cnv_rep_key is None:
        return None

    X = adata.obsm[cnv_rep_key]

    if sp.issparse(X):
        score = np.asarray(
            np.abs(X).mean(axis=1)
        ).ravel()
    else:
        score = np.nanmean(
            np.abs(np.asarray(X)),
            axis=1,
        )

    adata.obs[score_key] = score

    return score_key


def find_existing_col(adata, candidates):
    for c in candidates:
        if c in adata.obs.columns:
            return c
    return None


def infer_strategy_from_path(path):
    s = str(path)

    for k in HISTORICAL_QC_STRATEGY_ORDER:
        if k in s:
            return k

    return Path(path).stem


def save_current_fig(
    path,
    dpi=180,
    bbox_inches="tight",
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if path.exists():
        path.unlink()

    plt.savefig(
        path,
        dpi=dpi,
        bbox_inches=bbox_inches,
    )

    print("[SAVED FIG]", path)
    plt.close("all")

    return str(path)


def resolve_strategy_source_h5ads():
    """
    Resolve exactly one published notebook-11 result H5AD per strategy.
    Prefer the canonical notebook-11 run manifest.
    """

    resolved = {}

    if RUN_MANIFEST_TSV.exists():
        run_manifest = pd.read_csv(
            RUN_MANIFEST_TSV,
            sep="\t",
        )

        if (
            "strategy" in run_manifest.columns
            and "result_h5ad" in run_manifest.columns
        ):
            for _, row in run_manifest.iterrows():
                strategy = str(row["strategy"])

                if strategy not in HISTORICAL_QC_STRATEGY_ORDER:
                    continue

                path = Path(row["result_h5ad"])

                if path.exists():
                    resolved[strategy] = path

    if len(resolved) < 5:
        infer_manifest = safe_read_tsv(
            INFERCNVPY_OUTPUT_MANIFEST_TSV,
            required=False,
        )

        if not infer_manifest.empty:
            for col in infer_manifest.columns:
                if not (
                    "path" in col.lower()
                    or "file" in col.lower()
                    or "h5ad" in col.lower()
                ):
                    continue

                for value in (
                    infer_manifest[col]
                    .dropna()
                    .astype(str)
                ):
                    if not value.endswith(".h5ad"):
                        continue

                    path = Path(value)

                    if not path.exists():
                        continue

                    strategy = infer_strategy_from_path(path)

                    if (
                        strategy in HISTORICAL_QC_STRATEGY_ORDER
                        and "result" in path.name.lower()
                    ):
                        resolved.setdefault(
                            strategy,
                            path,
                        )

    if len(resolved) < 5 and CNV_RESULT_DIR.exists():
        for path in CNV_RESULT_DIR.glob("*_result.h5ad"):
            strategy = infer_strategy_from_path(path)

            if strategy in HISTORICAL_QC_STRATEGY_ORDER:
                resolved.setdefault(
                    strategy,
                    path,
                )

    missing = [
        strategy
        for strategy
        in HISTORICAL_QC_STRATEGY_ORDER
        if strategy not in resolved
    ]

    if missing:
        raise FileNotFoundError(
            "Could not resolve published notebook-11 result H5ADs for: "
            f"{missing}"
        )

    return [
        resolved[strategy]
        for strategy
        in HISTORICAL_QC_STRATEGY_ORDER
    ]


def _cache_is_valid(
    cache_path,
    source_path,
    expected_cells,
):
    cache_path = Path(cache_path)
    source_path = Path(source_path)

    if not cache_path.exists():
        return False

    check = sc.read_h5ad(
        cache_path,
        backed="r",
    )

    try:
        if check.shape != (
            expected_cells,
            EXPECTED_N_GENES,
        ):
            return False

        if "X_cnv" not in check.obsm:
            return False

        if tuple(check.obsm["X_cnv"].shape) != (
            expected_cells,
            EXPECTED_N_CNV_WINDOWS,
        ):
            return False

        if "X_umap" not in check.obsm:
            return False

        if str(
            check.uns.get(
                "visual_qc_source_h5ad",
                "",
            )
        ) != str(source_path):
            return False

        if int(
            check.uns.get(
                "visual_qc_source_size",
                -1,
            )
        ) != int(source_path.stat().st_size):
            return False

        return True

    finally:
        check.file.close()


def build_lightweight_cache(source_path):
    """
    Build/reuse an NVMe H5AD containing only the data needed for notebook 12:
    obs, var, uns, X_cnv, and X_umap.

    The source expression matrix X is never read.
    """

    source_path = Path(source_path)
    strategy = infer_strategy_from_path(source_path)
    expected_cells = EXPECTED_STRATEGY_CELLS[strategy]

    cache_path = (
        SCRATCH_ROOT
        / f"visual_qc_light_{strategy}_{RUN_SUFFIX}.h5ad"
    )

    if _cache_is_valid(
        cache_path,
        source_path,
        expected_cells,
    ):
        print(
            "[REUSING LIGHTWEIGHT CACHE]",
            cache_path,
        )

        with h5py.File(source_path, "r") as handle:
            source_obsm_keys = list(
                handle["obsm"].keys()
            )

        return (
            cache_path,
            source_obsm_keys,
        )

    if cache_path.exists():
        cache_path.unlink()

    print(
        "[BUILD LIGHTWEIGHT CACHE]",
        strategy,
    )

    print(source_path)
    print("->")
    print(cache_path)

    with h5py.File(source_path, "r") as handle:

        obs = read_elem(handle["obs"])
        var = read_elem(handle["var"])

        uns = (
            read_elem(handle["uns"])
            if "uns" in handle
            else {}
        )

        if (
            "obsm" not in handle
            or "X_cnv" not in handle["obsm"]
        ):
            raise KeyError(
                f"{source_path} is missing /obsm/X_cnv."
            )

        X_cnv = read_elem(
            handle["obsm"]["X_cnv"]
        )

        X_umap = (
            read_elem(handle["obsm"]["X_umap"])
            if "X_umap" in handle["obsm"]
            else None
        )

        source_obsm_keys = list(
            handle["obsm"].keys()
        )

    light = ad.AnnData(
        X=None,
        obs=obs.copy(),
        var=var.copy(),
        uns=copy.deepcopy(uns),
    )

    light.obsm["X_cnv"] = X_cnv

    if X_umap is not None:
        light.obsm["X_umap"] = X_umap

    light.uns["visual_qc_source_h5ad"] = str(
        source_path
    )

    light.uns["visual_qc_source_size"] = int(
        source_path.stat().st_size
    )

    light.uns["visual_qc_memory_safe_cache"] = True

    light.write_h5ad(
        cache_path,
        compression="gzip",
    )

    del light
    del X_cnv
    del X_umap
    gc.collect()

    if not _cache_is_valid(
        cache_path,
        source_path,
        expected_cells,
    ):
        raise RuntimeError(
            f"Lightweight cache validation failed: {cache_path}"
        )

    return (
        cache_path,
        source_obsm_keys,
    )

In [3]:
# =========================
# Load previous summary tables
# =========================

combined_run_summary = safe_read_tsv(COMBINED_RUN_SUMMARY_TSV)
combined_cluster_summary = safe_read_tsv(COMBINED_CLUSTER_SUMMARY_TSV)
consensus_calls = safe_read_tsv(CONSENSUS_CALLS_TSV)
infer_manifest = safe_read_tsv(INFERCNVPY_OUTPUT_MANIFEST_TSV)

print("combined_run_summary:", combined_run_summary.shape)
print("combined_cluster_summary:", combined_cluster_summary.shape)
print("consensus_calls:", consensus_calls.shape)
print("infer_manifest:", infer_manifest.shape)

if not consensus_calls.empty:
    display(consensus_calls)

if not combined_run_summary.empty:
    display(combined_run_summary)


# Historical notebook-11 input checkpoints.
if combined_run_summary.shape != EXPECTED_COMBINED_RUN_SHAPE:
    raise AssertionError(
        f"combined_run_summary changed: {combined_run_summary.shape}"
    )

if combined_cluster_summary.shape != EXPECTED_COMBINED_CLUSTER_SHAPE:
    raise AssertionError(
        f"combined_cluster_summary changed: {combined_cluster_summary.shape}"
    )

if consensus_calls.shape != EXPECTED_CONSENSUS_SHAPE:
    raise AssertionError(
        f"consensus_calls changed: {consensus_calls.shape}"
    )

if infer_manifest.shape != EXPECTED_INFER_MANIFEST_SHAPE:
    raise AssertionError(
        f"infer_manifest changed: {infer_manifest.shape}"
    )

if set(consensus_calls["cluster"].astype(str)) != EXPECTED_QUERY_REVIEW_CLUSTERS:
    raise RuntimeError(
        "Query/review cluster set differs from the archived run."
    )

if not (
    consensus_calls["consensus_cnv_call"].astype(str)
    == "CNV_low_or_reference_like"
).all():
    raise RuntimeError(
        "Notebook-11 infercnvpy consensus differs from the archived article run."
    )

print(
    "\n[NOTEBOOK-11 CANONICAL CONSENSUS VALIDATED]"
)


combined_run_summary: (5, 18)
combined_cluster_summary: (30, 30)
consensus_calls: (7, 11)
infer_manifest: (31, 4)


,cluster,n_strategies_observed,n_high_calls,n_intermediate_or_high_calls,strategies_high,strategies_intermediate_or_high,median_of_strategy_median_cnv_abs_score,max_robust_effect_vs_reference_median,consensus_cnv_call,top_annotation_marker_based_HVG8000_r0_6,top_lineage_marker_based_HVG8000_r0_6
0,11,3,0,0,NaN,NaN,0.017346,1.182955,CNV_low_or_reference_like,putative_tumor_NPC_neuronal_like,putative_tumor_like
1,0,3,0,0,NaN,NaN,0.016515,0.995507,CNV_low_or_reference_like,putative_tumor_OPC_NPC_like,putative_tumor_like
2,7,3,0,0,NaN,NaN,0.015950,0.880782,CNV_low_or_reference_like,putative_tumor_AC_like_reactive,putative_tumor_like
3,2,1,0,0,NaN,NaN,0.016681,0.823502,CNV_low_or_reference_like,cycling_proliferating_cells,cycling_ambiguous
4,4,3,0,0,NaN,NaN,0.012362,0.295791,CNV_low_or_reference_like,putative_tumor_hypoxia_stress,putative_tumor_like_stress
5,12,1,0,0,NaN,NaN,0.013881,0.115639,CNV_low_or_reference_like,MES_ECM_stromal_like_ambiguous,mesenchymal_stromal_ambiguous
6,9,1,0,0,NaN,NaN,0.013401,-0.000631,CNV_low_or_reference_like,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet


,strategy,input_h5ad,result_h5ad,cell_scores_tsv,cluster_summary_tsv,cnv_leiden_summary_tsv,n_cells,n_genes,n_cnv_windows,preprocessing,elapsed_minutes,figures,high_threshold,moderate_threshold,reference_mad,reference_median,reference_q95,reference_q99
0,core_conservative,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,32640,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,NaN,NaN,0.025629,0.021525,0.002768,0.013318,0.020419,0.024693
1,core_plus_lymphoid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,38807,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,3.398,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.025536,0.021259,0.002885,0.012704,0.020549,0.025131
2,extended_myeloid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,45000,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,2.442,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.030094,0.024566,0.004394,0.010548,0.024566,0.029647
3,review_cycling,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,14640,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,0.727,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.025605,0.021505,0.002766,0.013304,0.020389,0.024693
4,review_ambiguous,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,19443,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,0.956,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.025773,0.021650,0.002781,0.013404,0.020553,0.024805



[NOTEBOOK-11 CANONICAL CONSENSUS VALIDATED]


In [4]:
# =========================
# Discover infercnvpy result h5ad files + build NVMe lightweight caches
# =========================

strategy_source_h5ads = resolve_strategy_source_h5ads()

discovery = pd.DataFrame(
    {
        "path": [
            str(p)
            for p
            in strategy_source_h5ads
        ],
        "strategy_guess": [
            infer_strategy_from_path(p)
            for p
            in strategy_source_h5ads
        ],
        "exists": [
            p.exists()
            for p
            in strategy_source_h5ads
        ],
        "size_MB": [
            round(
                p.stat().st_size
                / 1024**2,
                2,
            )
            for p
            in strategy_source_h5ads
        ],
    }
)

display(discovery)

if discovery.shape != EXPECTED_DISCOVERY_SHAPE:
    raise AssertionError(
        f"Expected five strategy results, got {discovery.shape}"
    )

if discovery["strategy_guess"].tolist() != HISTORICAL_QC_STRATEGY_ORDER:
    raise RuntimeError(
        "Strategy discovery order differs from the archived QC run."
    )


strategy_h5ads = []
SOURCE_BY_STRATEGY = {}
SOURCE_OBSM_KEYS_BY_STRATEGY = {}

for source_path in strategy_source_h5ads:
    strategy = infer_strategy_from_path(source_path)

    cache_path, source_obsm_keys = build_lightweight_cache(
        source_path
    )

    strategy_h5ads.append(
        cache_path
    )

    SOURCE_BY_STRATEGY[strategy] = source_path
    SOURCE_OBSM_KEYS_BY_STRATEGY[strategy] = source_obsm_keys


print(
    "\n[ALL FIVE LIGHTWEIGHT VISUAL-QC CACHES READY]"
)

,path,strategy_guess,exists,size_MB
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,core_conservative,True,621.17
1,G:\Repository\glioma-cnv-single-cell-atlas\dat...,core_plus_lymphoid,True,705.50
2,G:\Repository\glioma-cnv-single-cell-atlas\dat...,extended_myeloid,True,835.53
3,G:\Repository\glioma-cnv-single-cell-atlas\dat...,review_ambiguous,True,320.10
4,G:\Repository\glioma-cnv-single-cell-atlas\dat...,review_cycling,True,315.99


[BUILD LIGHTWEIGHT CACHE] core_conservative
G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\infercnvpy_HVG8000_r0_6\infercnvpy_core_conservative_HVG8000_r0_6_result.h5ad
->
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_core_conservative_HVG8000_r0_6.h5ad
[BUILD LIGHTWEIGHT CACHE] core_plus_lymphoid
G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\infercnvpy_HVG8000_r0_6\infercnvpy_core_plus_lymphoid_HVG8000_r0_6_result.h5ad
->
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_core_plus_lymphoid_HVG8000_r0_6.h5ad
[BUILD LIGHTWEIGHT CACHE] extended_myeloid
G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\infercnvpy_HVG8000_r0_6\infercnvpy_extended_myeloid_HVG8000_r0_6_result.h5ad
->
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_extended_myeloid_HVG8000_r0_6.h5ad
[BUILD LIGHTWEIGHT CACHE] review_ambiguous
G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_r

In [5]:
# =========================
# Inspect each infercnvpy result object
# =========================

inspection_rows = []

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    print("\n" + "="*90)
    print(strategy)
    print(p)

    adata_cnv = sc.read_h5ad(p)

    if adata_cnv.shape != (
        EXPECTED_STRATEGY_CELLS[strategy],
        EXPECTED_N_GENES,
    ):
        raise AssertionError(
            f"{strategy} lightweight cache shape changed: {adata_cnv.shape}"
        )

    if "X_cnv" not in adata_cnv.obsm:
        raise KeyError(
            f"{strategy} cache is missing X_cnv."
        )

    if tuple(adata_cnv.obsm["X_cnv"].shape) != (
        EXPECTED_STRATEGY_CELLS[strategy],
        EXPECTED_N_CNV_WINDOWS,
    ):
        raise AssertionError(
            f"{strategy} X_cnv shape changed: "
            f"{adata_cnv.obsm['X_cnv'].shape}"
        )
    cnv_rep = detect_cnv_rep(adata_cnv)
    score_col = detect_cnv_score_col(adata_cnv)
    score_col_final = score_col if score_col is not None else ensure_cnv_score(adata_cnv, cnv_rep)

    role_col = find_existing_col(adata_cnv, ["CNV_role", "cnv_role", "CNV_category", "cnv_category"])
    cluster_col = find_existing_col(adata_cnv, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6"])
    annotation_col = find_existing_col(adata_cnv, ["annotation_marker_based_HVG8000_r0_6", "annotation_marker_based", "annotation"])
    cnv_leiden_col = find_existing_col(adata_cnv, ["cnv_leiden", "CNV_leiden", "infercnvpy_leiden"])

    if score_col_final is not None:
        score_stats = finite_summary(adata_cnv.obs[score_col_final].astype(float).values)
    else:
        score_stats = finite_summary([])

    inspection_rows.append({
        "strategy": strategy,
        "h5ad_path": str(SOURCE_BY_STRATEGY[strategy]),
        "n_cells": adata_cnv.n_obs,
        "n_genes": adata_cnv.n_vars,
        "obsm_keys": ";".join(map(str, SOURCE_OBSM_KEYS_BY_STRATEGY[strategy])),
        "obs_columns_n": len(adata_cnv.obs.columns),
        "var_columns": ";".join(map(str, adata_cnv.var.columns)),
        "cnv_rep_key": cnv_rep,
        "cnv_score_col": score_col_final,
        "role_col": role_col,
        "cluster_col": cluster_col,
        "annotation_col": annotation_col,
        "cnv_leiden_col": cnv_leiden_col,
        **{f"cnv_score_{k}": v for k, v in score_stats.items()},
    })

    del adata_cnv
    gc.collect()

inspection_df = pd.DataFrame(inspection_rows)

if inspection_df.shape != EXPECTED_STRATEGY_VISUAL_SHAPE:
    raise AssertionError(
        f"Strategy visual summary changed: {inspection_df.shape}"
    )

write_tsv_replace(inspection_df, STRATEGY_VISUAL_SUMMARY_TSV)
display(inspection_df)



core_conservative
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_core_conservative_HVG8000_r0_6.h5ad

core_plus_lymphoid
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_core_plus_lymphoid_HVG8000_r0_6.h5ad

extended_myeloid
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_extended_myeloid_HVG8000_r0_6.h5ad

review_ambiguous
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_review_ambiguous_HVG8000_r0_6.h5ad

review_cycling
E:\glioma-cnv-scratch\main_analysis_12\visual_qc_light_review_cycling_HVG8000_r0_6.h5ad
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_visual_QC_strategy_summary_HVG8000_r0_6.tsv shape=(5, 22)


,strategy,h5ad_path,n_cells,n_genes,obsm_keys,obs_columns_n,var_columns,cnv_rep_key,cnv_score_col,role_col,...,cnv_leiden_col,cnv_score_min,cnv_score_q05,cnv_score_q25,cnv_score_median,cnv_score_mean,cnv_score_q75,cnv_score_q95,cnv_score_max,cnv_score_sd
0,core_conservative,G:\Repository\glioma-cnv-single-cell-atlas\dat...,32640,15163,X_cnv;X_cnv_pca;X_cnv_umap;X_scVI;X_umap;cnv_umap,77,cnv_chr;cnv_start;cnv_end;chromosome;start;end,X_cnv,cnv_abs_mean_score,None,...,cnv_leiden,0.001366,0.007677,0.011589,0.014863,0.015352,0.018438,0.024531,0.123524,0.005465
1,core_plus_lymphoid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,38807,15163,X_cnv;X_cnv_pca;X_cnv_umap;X_scVI;X_umap;cnv_umap,77,cnv_chr;cnv_start;cnv_end;chromosome;start;end,X_cnv,cnv_abs_mean_score,None,...,cnv_leiden,0.000120,0.007044,0.011048,0.014310,0.014919,0.018093,0.024683,0.123157,0.005696
2,extended_myeloid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,45000,15163,X_cnv;X_cnv_pca;X_cnv_umap;X_scVI;X_umap;cnv_umap,77,cnv_chr;cnv_start;cnv_end;chromosome;start;end,X_cnv,cnv_abs_mean_score,None,...,cnv_leiden,0.000692,0.005061,0.009625,0.014224,0.014744,0.019152,0.026140,0.122312,0.006722
3,review_ambiguous,G:\Repository\glioma-cnv-single-cell-atlas\dat...,19443,15163,X_cnv;X_cnv_pca;X_cnv_umap;X_scVI;X_umap;cnv_umap,77,cnv_chr;cnv_start;cnv_end;chromosome;start;end,X_cnv,cnv_abs_mean_score,None,...,cnv_leiden,0.001366,0.007416,0.011222,0.013494,0.013757,0.015915,0.020895,0.071289,0.004139
4,review_cycling,G:\Repository\glioma-cnv-single-cell-atlas\dat...,14640,15163,X_cnv;X_cnv_pca;X_cnv_umap;X_scVI;X_umap;cnv_umap,77,cnv_chr;cnv_start;cnv_end;chromosome;start;end,X_cnv,cnv_abs_mean_score,None,...,cnv_leiden,0.001047,0.007252,0.011391,0.014706,0.014922,0.018074,0.023260,0.055692,0.005019


In [6]:
# =========================
# Plot transcriptomic UMAPs per strategy
# =========================

figure_records = []
cluster_score_records = []

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    adata_cnv = sc.read_h5ad(p)

    cnv_rep = detect_cnv_rep(adata_cnv)
    score_col = detect_cnv_score_col(adata_cnv)
    if score_col is None:
        score_col = ensure_cnv_score(adata_cnv, cnv_rep)

    role_col = find_existing_col(adata_cnv, ["CNV_role", "cnv_role", "CNV_category", "cnv_category"])
    cluster_col = find_existing_col(adata_cnv, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6"])
    annotation_col = find_existing_col(adata_cnv, ["annotation_marker_based_HVG8000_r0_6", "annotation_marker_based", "annotation"])
    cnv_leiden_col = find_existing_col(adata_cnv, ["cnv_leiden", "CNV_leiden", "infercnvpy_leiden"])

    if "X_umap" not in adata_cnv.obsm:
        print(f"[SKIP UMAP] X_umap not found for {strategy}")
        continue

    colors = [c for c in [score_col, role_col, cluster_col, annotation_col, cnv_leiden_col] if c is not None and c in adata_cnv.obs.columns]
    if len(colors) == 0:
        continue

    for color in colors:
        try:
            sc.pl.umap(
                adata_cnv,
                color=color,
                show=False,
                frameon=False,
                size=4 if adata_cnv.n_obs < 50000 else 2,
                title=f"{strategy} — {color}",
            )
            out = CNV_FIG_DIR / f"UMAP_{sanitize_name(strategy)}_{sanitize_name(color)}.png"
            save_current_fig(out)
            figure_records.append({
                "strategy": strategy,
                "figure_type": "transcriptomic_umap",
                "color": color,
                "path": str(out),
            })
        except Exception as e:
            print(f"[WARNING] UMAP failed for {strategy} {color}: {repr(e)}")

    if cluster_col is not None and score_col is not None:
        tmp = adata_cnv.obs[[cluster_col, score_col]].copy()
        tmp[score_col] = tmp[score_col].astype(float)
        grp = tmp.groupby(cluster_col, observed=True)[score_col].agg(
            n_cells="size",
            cnv_score_mean="mean",
            cnv_score_median="median",
            cnv_score_q75=lambda x: np.quantile(x, 0.75),
            cnv_score_q95=lambda x: np.quantile(x, 0.95),
            cnv_score_max="max",
        ).reset_index().rename(columns={cluster_col: "cluster"})
        grp["strategy"] = strategy
        cluster_score_records.append(grp)

    del adata_cnv
    gc.collect()
    plt.close("all")

if len(cluster_score_records) > 0:

    cluster_score_df = pd.concat(cluster_score_records, ignore_index=True)
else:
    cluster_score_df = pd.DataFrame()

write_tsv_replace(cluster_score_df, CLUSTER_VISUAL_SUMMARY_TSV)
display(cluster_score_df.head(30))


if cluster_score_df.shape != EXPECTED_CLUSTER_SCORE_SHAPE:
    raise AssertionError(
        f"Cluster score summary changed: {cluster_score_df.shape}"
    )

# Internal consistency audit against notebook 11.
audit_visual = cluster_score_df.copy()
audit_visual["cluster"] = audit_visual["cluster"].astype(str)

audit_n11 = combined_cluster_summary.copy()
audit_n11["cluster"] = audit_n11["cluster"].astype(str)

score_audit = audit_visual.merge(
    audit_n11[
        [
            "strategy",
            "cluster",
            "n_cells",
            "mean_cnv_abs_score",
            "median_cnv_abs_score",
        ]
    ],
    on=[
        "strategy",
        "cluster",
    ],
    how="inner",
    suffixes=(
        "_visual",
        "_notebook11",
    ),
)

if score_audit.shape[0] != 30:
    raise RuntimeError(
        "Could not align all 30 visual-QC cluster rows to notebook 11."
    )

if not (
    score_audit["n_cells_visual"].astype(int)
    == score_audit["n_cells_notebook11"].astype(int)
).all():
    raise RuntimeError(
        "Visual-QC cell counts differ from notebook 11."
    )

if not np.allclose(
    score_audit["cnv_score_mean"],
    score_audit["mean_cnv_abs_score"],
    rtol=1e-10,
    atol=1e-12,
):
    raise RuntimeError(
        "Visual-QC mean CNV scores differ from notebook 11."
    )

if not np.allclose(
    score_audit["cnv_score_median"],
    score_audit["median_cnv_abs_score"],
    rtol=1e-10,
    atol=1e-12,
):
    raise RuntimeError(
        "Visual-QC median CNV scores differ from notebook 11."
    )

print(
    "[INTERNAL SCORE AUDIT PASSED: 30/30 biological-cluster rows]"
)


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_conservative_cnv_abs_mean_score.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_conservative_leiden_scvi_main.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_conservative_annotation_marker_based_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_conservative_cnv_leiden.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_plus_lymphoid_cnv_abs_mean_score.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_plus_lymphoid_leiden_scvi_main.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\UMAP_core_plus_lymphoid_annotation_marker_based_HVG8000_r0_6.png
[SAVED FIG] G:\Reposi

,cluster,n_cells,cnv_score_mean,cnv_score_median,cnv_score_q75,cnv_score_q95,cnv_score_max,strategy
0,0,6387,0.016490,0.016386,0.019710,0.024920,0.056345,core_conservative
1,4,6432,0.013421,0.012362,0.016055,0.022753,0.123524,core_conservative
2,7,6479,0.016166,0.015603,0.019086,0.025253,0.056751,core_conservative
3,8,5000,0.013245,0.013126,0.015738,0.019971,0.033014,core_conservative
4,11,5702,0.017953,0.017346,0.020821,0.027124,0.060969,core_conservative
5,14,2640,0.013675,0.013792,0.016613,0.021080,0.038176,core_conservative
6,0,6387,0.016714,0.016515,0.020000,0.025810,0.056379,core_plus_lymphoid
7,4,6432,0.013227,0.012074,0.015867,0.023251,0.123157,core_plus_lymphoid
8,5,5000,0.012108,0.011790,0.014245,0.018800,0.032817,core_plus_lymphoid
9,7,6479,0.016620,0.015950,0.019664,0.026611,0.058500,core_plus_lymphoid


[INTERNAL SCORE AUDIT PASSED: 30/30 biological-cluster rows]


In [7]:
# =========================
# infercnvpy chromosome heatmaps
# =========================

if not HAS_INFERCNVPY:
    print("[SKIP] infercnvpy unavailable.")
else:
    for p in strategy_h5ads:
        strategy = infer_strategy_from_path(p)
        adata_cnv = sc.read_h5ad(p)

        cnv_rep_key = detect_cnv_rep(adata_cnv)
        if cnv_rep_key is None:
            print(f"[SKIP] no CNV representation for {strategy}")
            continue

        if cnv_rep_key.startswith("X_"):
            use_rep = cnv_rep_key[2:]
        else:
            use_rep = cnv_rep_key

        group_candidates = [
            "cnv_leiden",
            "leiden_scvi_main",
            "CNV_role",
            "cnv_role",
            "annotation_marker_based_HVG8000_r0_6",
        ]
        group_cols = [g for g in group_candidates if g in adata_cnv.obs.columns]

        for groupby in group_cols:
            try:
                cnv.pl.chromosome_heatmap(
                    adata_cnv,
                    groupby=groupby,
                    use_rep=use_rep,
                    show=False,
                )
                out = CNV_FIG_DIR / f"infercnvpy_chromosome_heatmap_{sanitize_name(strategy)}_groupby_{sanitize_name(groupby)}.png"
                save_current_fig(out, dpi=160)
                figure_records.append({
                    "strategy": strategy,
                    "figure_type": "infercnvpy_chromosome_heatmap",
                    "groupby": groupby,
                    "path": str(out),
                })
            except Exception as e:
                print(f"[WARNING] chromosome_heatmap failed for {strategy} groupby={groupby}: {repr(e)}")

        for groupby in group_cols:
            try:
                cnv.pl.chromosome_heatmap_summary(
                    adata_cnv,
                    groupby=groupby,
                    use_rep=use_rep,
                    show=False,
                )
                out = CNV_FIG_DIR / f"infercnvpy_chromosome_heatmap_summary_{sanitize_name(strategy)}_groupby_{sanitize_name(groupby)}.png"
                save_current_fig(out, dpi=160)
                figure_records.append({
                    "strategy": strategy,
                    "figure_type": "infercnvpy_chromosome_heatmap_summary",
                    "groupby": groupby,
                    "path": str(out),
                })
            except Exception as e:
                print(f"[WARNING] chromosome_heatmap_summary failed for {strategy} groupby={groupby}: {repr(e)}")

        del adata_cnv
        gc.collect()
        plt.close("all")


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\infercnvpy_chromosome_heatmap_core_conservative_groupby_cnv_leiden.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\infercnvpy_chromosome_heatmap_core_conservative_groupby_leiden_scvi_main.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\infercnvpy_chromosome_heatmap_core_conservative_groupby_annotation_marker_based_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\infercnvpy_chromosome_heatmap_summary_core_conservative_groupby_cnv_leiden.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\infercnvpy_chromosome_heatmap_summary_core_conservative_groupby_leiden_scvi_main.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\infercnvpy_chromosome_heatmap_summary_core_conserva

In [8]:
# =========================
# Custom group-level CNV heatmaps from X_cnv
# =========================

def group_mean_matrix(adata, rep_key, groupby, max_groups=80):
    if rep_key is None or groupby not in adata.obs.columns:
        return None, None

    X = adata.obsm[rep_key]
    groups = adata.obs[groupby].astype(str)
    group_order = groups.value_counts().index.tolist()
    if len(group_order) > max_groups:
        group_order = group_order[:max_groups]

    rows = []
    names = []
    for g in group_order:
        idx = np.where(groups.values == g)[0]
        if len(idx) == 0:
            continue
        Xi = X[idx]
        if sp.issparse(Xi):
            mean = np.asarray(Xi.mean(axis=0)).ravel()
        else:
            mean = np.nanmean(np.asarray(Xi), axis=0)
        rows.append(mean)
        names.append(g)

    if len(rows) == 0:
        return None, None

    return np.vstack(rows), names

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    adata_cnv = sc.read_h5ad(p)
    rep_key = detect_cnv_rep(adata_cnv)

    if rep_key is None:
        continue

    groupby_candidates = [
        "leiden_scvi_main",
        "cnv_leiden",
        "CNV_role",
        "cnv_role",
        "annotation_marker_based_HVG8000_r0_6",
    ]
    groupby_candidates = [g for g in groupby_candidates if g in adata_cnv.obs.columns]

    for groupby in groupby_candidates:
        M, names = group_mean_matrix(adata_cnv, rep_key, groupby)
        if M is None:
            continue

        vmax = np.nanquantile(np.abs(M), 0.98)
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 1.0

        plt.figure(figsize=(16, max(4, 0.28 * len(names))))
        plt.imshow(M, aspect="auto", cmap="bwr", vmin=-vmax, vmax=vmax, interpolation="nearest")
        plt.yticks(np.arange(len(names)), names, fontsize=8)
        plt.xlabel("CNV genomic window/bin")
        plt.ylabel(groupby)
        plt.title(f"{strategy} — mean CNV profile by {groupby}")
        plt.colorbar(label="mean CNV signal")
        out = CNV_FIG_DIR / f"custom_mean_CNV_heatmap_{sanitize_name(strategy)}_groupby_{sanitize_name(groupby)}.png"
        save_current_fig(out, dpi=160)
        figure_records.append({
            "strategy": strategy,
            "figure_type": "custom_mean_CNV_heatmap",
            "groupby": groupby,
            "path": str(out),
        })

    del adata_cnv
    gc.collect()
    plt.close("all")


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\custom_mean_CNV_heatmap_core_conservative_groupby_leiden_scvi_main.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\custom_mean_CNV_heatmap_core_conservative_groupby_cnv_leiden.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\custom_mean_CNV_heatmap_core_conservative_groupby_annotation_marker_based_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\custom_mean_CNV_heatmap_core_plus_lymphoid_groupby_leiden_scvi_main.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\custom_mean_CNV_heatmap_core_plus_lymphoid_groupby_cnv_leiden.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6\custom_mean_CNV_heatmap_core_plus_lymphoid_groupby_annotation_marker_based_HVG8000_r0_6.png
[SA

In [9]:
# =========================
# Try chromosome-level summaries from CNV bins
# =========================

def extract_window_chromosomes(adata, rep_key):
    """Best-effort extraction of chromosome label for each CNV window."""
    n_bins = adata.obsm[rep_key].shape[1] if rep_key is not None else None
    if n_bins is None:
        return None

    candidate_uns_keys = []
    if rep_key.startswith("X_"):
        candidate_uns_keys.append(rep_key[2:])
    candidate_uns_keys.append(rep_key)
    candidate_uns_keys.extend([k for k in adata.uns.keys() if "cnv" in str(k).lower()])

    for uk in candidate_uns_keys:
        if uk not in adata.uns:
            continue
        obj = adata.uns[uk]

        if isinstance(obj, dict):
            for key in ["chr", "chromosome", "chromosomes", "chr_pos", "chromosome_pos", "window_chromosomes"]:
                if key in obj:
                    val = obj[key]
                    try:
                        arr = np.asarray(val)
                        if arr.ndim == 1 and len(arr) == n_bins:
                            return arr.astype(str)
                    except Exception:
                        pass

            for key, val in obj.items():
                try:
                    if isinstance(val, pd.DataFrame):
                        for col in ["chromosome", "chr", "cnv_chr"]:
                            if col in val.columns and len(val) == n_bins:
                                return val[col].astype(str).values
                except Exception:
                    pass

    return None

chrom_records = []

for p in strategy_h5ads:
    strategy = infer_strategy_from_path(p)
    adata_cnv = sc.read_h5ad(p)
    rep_key = detect_cnv_rep(adata_cnv)
    cluster_col = find_existing_col(adata_cnv, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6"])
    role_col = find_existing_col(adata_cnv, ["CNV_role", "cnv_role", "CNV_category", "cnv_category"])

    if rep_key is None:
        continue

    chroms = extract_window_chromosomes(adata_cnv, rep_key)
    if chroms is None:
        print(f"[INFO] Could not extract chromosome per CNV window for {strategy}; skipping chromosome-level signal table.")
        continue

    X = adata_cnv.obsm[rep_key]
    group_cols = [c for c in [cluster_col, role_col, "cnv_leiden"] if c is not None and c in adata_cnv.obs.columns]

    for group_col in group_cols:
        groups = adata_cnv.obs[group_col].astype(str)

        for group in sorted(groups.unique(), key=lambda x: str(x)):
            idx = np.where(groups.values == group)[0]
            if len(idx) == 0:
                continue
            Xi = X[idx]
            if sp.issparse(Xi):
                mean_bin = np.asarray(Xi.mean(axis=0)).ravel()
            else:
                mean_bin = np.nanmean(np.asarray(Xi), axis=0)

            for chrom in pd.unique(chroms):
                j = np.where(chroms == chrom)[0]
                if len(j) == 0:
                    continue
                vals = mean_bin[j]
                chrom_records.append({
                    "strategy": strategy,
                    "groupby": group_col,
                    "group": group,
                    "chromosome": chrom,
                    "n_windows": int(len(j)),
                    "mean_cnv_signal": float(np.nanmean(vals)),
                    "median_cnv_signal": float(np.nanmedian(vals)),
                    "mean_abs_cnv_signal": float(np.nanmean(np.abs(vals))),
                    "max_abs_cnv_signal": float(np.nanmax(np.abs(vals))),
                })

    del adata_cnv
    gc.collect()

chrom_df = pd.DataFrame(chrom_records)

if chrom_df.shape != EXPECTED_CHROMOSOME_SIGNAL_SHAPE:
    raise RuntimeError(
        "Historical chromosome-window extraction behavior changed: "
        f"{chrom_df.shape}; archived execution produced (0, 0)."
    )

write_tsv_replace(chrom_df, CHROMOSOME_SIGNAL_TSV)
display(chrom_df.head(30))


[INFO] Could not extract chromosome per CNV window for core_conservative; skipping chromosome-level signal table.
[INFO] Could not extract chromosome per CNV window for core_plus_lymphoid; skipping chromosome-level signal table.
[INFO] Could not extract chromosome per CNV window for extended_myeloid; skipping chromosome-level signal table.
[INFO] Could not extract chromosome per CNV window for review_ambiguous; skipping chromosome-level signal table.
[INFO] Could not extract chromosome per CNV window for review_cycling; skipping chromosome-level signal table.
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_visual_QC_chromosome_signal_summary_HVG8000_r0_6.tsv shape=(0, 0)


""


In [10]:
# =========================
# Build interpretation table
# =========================

if not cluster_score_df.empty:
    score_pivot = cluster_score_df.copy()
    score_pivot["cluster"] = score_pivot["cluster"].astype(str)

    agg = score_pivot.groupby("cluster", observed=True).agg(
        n_strategies_observed=("strategy", "nunique"),
        max_cluster_median_cnv_score=("cnv_score_median", "max"),
        median_cluster_median_cnv_score=("cnv_score_median", "median"),
        max_cluster_q95_cnv_score=("cnv_score_q95", "max"),
        max_cluster_max_cnv_score=("cnv_score_max", "max"),
    ).reset_index()
else:
    agg = pd.DataFrame(columns=[
        "cluster", "n_strategies_observed", "max_cluster_median_cnv_score",
        "median_cluster_median_cnv_score", "max_cluster_q95_cnv_score", "max_cluster_max_cnv_score"
    ])

if not consensus_calls.empty:
    cc = consensus_calls.copy()
    if "cluster" not in cc.columns:
        possible = [c for c in cc.columns if "cluster" in c.lower()]
        if possible:
            cc = cc.rename(columns={possible[0]: "cluster"})
    cc["cluster"] = cc["cluster"].astype(str)
    interp = cc.merge(agg, on="cluster", how="left")
else:
    interp = agg.copy()

def interpret_row(row):
    max_med = row.get("max_cluster_median_cnv_score", np.nan)
    max_q95 = row.get("max_cluster_q95_cnv_score", np.nan)
    consensus = str(row.get("consensus_call", row.get("CNV_consensus_call", ""))).lower()

    if "high" in consensus:
        return "visual_review_priority_high_consensus_cnv_high"
    if np.isfinite(max_med) and max_med >= 0.15:
        return "visual_review_priority_possible_broad_cnv"
    if np.isfinite(max_q95) and max_q95 >= 0.30:
        return "visual_review_priority_subpopulation_possible_cnv"
    if "low" in consensus or "reference" in consensus:
        return "currently_reference_like_by_score_visual_check_needed"
    return "insufficient_information_visual_check_needed"

if not interp.empty:
    interp["visual_interpretation_flag"] = interp.apply(interpret_row, axis=1)

write_tsv_replace(interp, INTERPRETATION_TSV)
display(interp)


if interp.shape != EXPECTED_INTERPRETATION_SHAPE:
    raise AssertionError(
        f"Interpretation table changed: {interp.shape}"
    )

if set(interp["cluster"].astype(str)) != EXPECTED_QUERY_REVIEW_CLUSTERS:
    raise RuntimeError(
        "Visual interpretation cluster set changed."
    )

if set(
    interp["visual_interpretation_flag"].astype(str)
) != {
    "insufficient_information_visual_check_needed"
}:
    raise RuntimeError(
        "Historical cautious visual interpretation flags changed."
    )

print(
    "\n[HISTORICAL CAUTIOUS VISUAL INTERPRETATION REPRODUCED]"
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_visual_QC_interpretation_table_HVG8000_r0_6.tsv shape=(7, 17)


,cluster,n_strategies_observed_x,n_high_calls,n_intermediate_or_high_calls,strategies_high,strategies_intermediate_or_high,median_of_strategy_median_cnv_abs_score,max_robust_effect_vs_reference_median,consensus_cnv_call,top_annotation_marker_based_HVG8000_r0_6,top_lineage_marker_based_HVG8000_r0_6,n_strategies_observed_y,max_cluster_median_cnv_score,median_cluster_median_cnv_score,max_cluster_q95_cnv_score,max_cluster_max_cnv_score,visual_interpretation_flag
0,11,3,0,0,NaN,NaN,0.017346,1.182955,CNV_low_or_reference_like,putative_tumor_NPC_neuronal_like,putative_tumor_like,3,0.018256,0.017346,0.028960,0.063409,insufficient_information_visual_check_needed
1,0,3,0,0,NaN,NaN,0.016515,0.995507,CNV_low_or_reference_like,putative_tumor_OPC_NPC_like,putative_tumor_like,3,0.017034,0.016515,0.027069,0.056379,insufficient_information_visual_check_needed
2,7,3,0,0,NaN,NaN,0.015950,0.880782,CNV_low_or_reference_like,putative_tumor_AC_like_reactive,putative_tumor_like,3,0.016287,0.015950,0.027307,0.060126,insufficient_information_visual_check_needed
3,2,1,0,0,NaN,NaN,0.016681,0.823502,CNV_low_or_reference_like,cycling_proliferating_cells,cycling_ambiguous,1,0.016681,0.016681,0.024955,0.055692,insufficient_information_visual_check_needed
4,4,3,0,0,NaN,NaN,0.012362,0.295791,CNV_low_or_reference_like,putative_tumor_hypoxia_stress,putative_tumor_like_stress,3,0.012476,0.012362,0.023948,0.123524,insufficient_information_visual_check_needed
5,12,1,0,0,NaN,NaN,0.013881,0.115639,CNV_low_or_reference_like,MES_ECM_stromal_like_ambiguous,mesenchymal_stromal_ambiguous,1,0.013881,0.013881,0.024167,0.071289,insufficient_information_visual_check_needed
6,9,1,0,0,NaN,NaN,0.013401,-0.000631,CNV_low_or_reference_like,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet,1,0.013401,0.013401,0.017999,0.049261,insufficient_information_visual_check_needed



[HISTORICAL CAUTIOUS VISUAL INTERPRETATION REPRODUCED]


In [11]:
# =========================
# Save figure and output manifests
# =========================

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, FIGURE_MANIFEST_TSV)

output_manifest = pd.DataFrame([
    {"file_type": "table", "description": "strategy-level visual QC summary", "path": str(STRATEGY_VISUAL_SUMMARY_TSV), "exists": STRATEGY_VISUAL_SUMMARY_TSV.exists()},
    {"file_type": "table", "description": "cluster-level CNV score summary across strategies", "path": str(CLUSTER_VISUAL_SUMMARY_TSV), "exists": CLUSTER_VISUAL_SUMMARY_TSV.exists()},
    {"file_type": "table", "description": "chromosome-level CNV signal summary when chromosome-bin metadata is available", "path": str(CHROMOSOME_SIGNAL_TSV), "exists": CHROMOSOME_SIGNAL_TSV.exists()},
    {"file_type": "table", "description": "cautious CNV visual interpretation table", "path": str(INTERPRETATION_TSV), "exists": INTERPRETATION_TSV.exists()},
    {"file_type": "table", "description": "figure manifest", "path": str(FIGURE_MANIFEST_TSV), "exists": FIGURE_MANIFEST_TSV.exists()},
    {"file_type": "folder", "description": "CNV visual QC figures", "path": str(CNV_FIG_DIR), "exists": CNV_FIG_DIR.exists()},
])
write_tsv_replace(output_manifest, OUTPUT_MANIFEST_TSV)
display(output_manifest)

print("\nImportant figure folder:")
print(CNV_FIG_DIR)


# Historical figure/output checkpoints.
for col in [
    "strategy",
    "figure_type",
    "color",
    "groupby",
    "path",
]:
    if col not in figure_manifest.columns:
        figure_manifest[col] = np.nan

figure_manifest = figure_manifest[
    [
        "strategy",
        "figure_type",
        "color",
        "groupby",
        "path",
    ]
]

if figure_manifest.shape != EXPECTED_FIGURE_MANIFEST_SHAPE:
    raise RuntimeError(
        "Historical figure manifest changed: "
        f"{figure_manifest.shape}"
    )

figure_type_counts = (
    figure_manifest["figure_type"]
    .value_counts()
    .to_dict()
)

expected_figure_type_counts = {
    "transcriptomic_umap": 20,
    "infercnvpy_chromosome_heatmap": 15,
    "infercnvpy_chromosome_heatmap_summary": 15,
    "custom_mean_CNV_heatmap": 15,
}

if figure_type_counts != expected_figure_type_counts:
    raise RuntimeError(
        "Historical figure-type counts changed: "
        f"{figure_type_counts}"
    )

if not figure_manifest["path"].map(
    lambda value: Path(value).exists()
).all():
    raise RuntimeError(
        "At least one expected visual-QC figure is missing."
    )

# Rewrite after schema normalization.
write_tsv_replace(
    figure_manifest,
    FIGURE_MANIFEST_TSV,
)

if output_manifest.shape != EXPECTED_OUTPUT_MANIFEST_SHAPE:
    raise AssertionError(
        f"Visual-QC output manifest changed: {output_manifest.shape}"
    )

if not output_manifest["exists"].all():
    raise RuntimeError(
        "At least one visual-QC output is missing."
    )

print(
    "\nDONE — canonical historical infercnvpy visual QC "
    "reproduced with memory-safe NVMe caches."
)

print(
    "Strategy visual summary:",
    inspection_df.shape,
)

print(
    "Cluster score summary:",
    cluster_score_df.shape,
)

print(
    "Chromosome signal summary:",
    chrom_df.shape,
)

print(
    "Interpretation table:",
    interp.shape,
)

print(
    "Figure manifest:",
    figure_manifest.shape,
)

print(
    "Figure type counts:",
    figure_type_counts,
)

print(
    "Output manifest:",
    output_manifest.shape,
)

print(
    "Scratch root:",
    SCRATCH_ROOT,
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_visual_QC_figure_manifest_HVG8000_r0_6.tsv shape=(65, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_visual_QC_output_manifest_HVG8000_r0_6.tsv shape=(6, 4)


,file_type,description,path,exists
0,table,strategy-level visual QC summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
1,table,cluster-level CNV score summary across strategies,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
2,table,chromosome-level CNV signal summary when chrom...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
3,table,cautious CNV visual interpretation table,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
4,table,figure manifest,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
5,folder,CNV visual QC figures,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True



Important figure folder:
G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_visual_QC_HVG8000_r0_6
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_visual_QC_figure_manifest_HVG8000_r0_6.tsv shape=(65, 5)

DONE — canonical historical infercnvpy visual QC reproduced with memory-safe NVMe caches.
Strategy visual summary: (5, 22)
Cluster score summary: (30, 8)
Chromosome signal summary: (0, 0)
Interpretation table: (7, 17)
Figure manifest: (65, 5)
Figure type counts: {'transcriptomic_umap': 20, 'infercnvpy_chromosome_heatmap': 15, 'infercnvpy_chromosome_heatmap_summary': 15, 'custom_mean_CNV_heatmap': 15}
Output manifest: (6, 4)
Scratch root: E:\glioma-cnv-scratch\main_analysis_12


## Expected successful completion

```text
Strategy visual summary:       (5, 22)
Cluster score summary:         (30, 8)
Chromosome signal summary:     (0, 0)
Interpretation table:          (7, 17)
Figure manifest:               (65, 5)
Output manifest:               (6, 4)
```

Expected figure counts:

```text
transcriptomic_umap                     20
infercnvpy_chromosome_heatmap           15
infercnvpy_chromosome_heatmap_summary   15
custom_mean_CNV_heatmap                 15
```

The final cell must also report:

```text
[INTERNAL SCORE AUDIT PASSED: 30/30 biological-cluster rows]
[HISTORICAL CAUTIOUS VISUAL INTERPRETATION REPRODUCED]
```

The scratch directory may be retained until the next CNV QC stage validates the
visual outputs.
